# 🤖 Notebook 4: Manipulator Control & Architecture Comparison

In this notebook, we transition from single-axis motor jogging to controlling a coordinated **6-DOF multi-axis robotic arm (Dobot CR3A)**. You will launch the manipulator stack, execute motion through WMX R2's **Proprietary Trajectory Control** path, and then compare it against **Standard ROS 2 Control**.

🖥️ **Simulation only, no Dobot CR3A on hand?** Skip Step 13's hardware launch commands and instead bring up the Gazebo stack described in `05_Visualization_and_Data_Logging.ipynb` (`gazebo_trajectory_simulation.launch.py` + `sim_bridge.launch.py`). Steps 14-15 below work identically once any Position-Mode-capable trajectory controller is running — real EtherCAT hardware or simulated.

*Prerequisite: Notebooks 01-03 completed (or the equivalent startup/Servo-On sequence performed for the manipulator's axes) so Position Mode is active.*

---

## 🔄 Understanding the Two Control Architectures

To control a complex manipulator, WMX R2 provides two distinct pathways with unique engineering trade-offs:

### 1. Proprietary Trajectory Control (`wmx_r2_package` Mode)
* **How it works**: `MoveIt 2` plans a path and sends a list of joint trajectory waypoints. The custom `joint_trajectory_controller` in `wmx_r2_package` intercepts these points and hands them directly to the **WMX3 core engine**. WMX3 then executes its proprietary **C-Spline interpolation algorithms** to smoothly drive the EtherCAT motors.
* **Key Benefit**: Extremely lightweight on CPU resources, and leverages decades of proven industrial-grade motion profiles with microsecond synchronization.

### 2. Standard ROS 2 Control (`wmx_r2_control` Mode)
* **How it works**: The execution loop is completely governed by the ROS 2 standard `controller_manager`. WMX3 is loaded purely as a low-level **Hardware Interface Plugin** (`WmxSystemHardware`). The standard ROS 2 `joint_trajectory_controller` reads encoder feedback via `read()` and injects raw target commands via `write()` at a real-time periodic rate.
* **Key Benefit**: Provides a **"Zero-Gap Digital Twin"** workflow — the high-level ROS 2 stack (MoveIt 2, RViz, Nav2, ...) cannot tell the difference between this real hardware and a Gazebo simulation, because both expose the exact same standard `ros2_control` hardware interface and the same `FollowJointTrajectory` action.

## Step 13: Launch the Manipulator Driver & Trajectory Stack

Just like Notebook 01's Step 1, the manipulator stack runs as background ROS 2 nodes in separate terminals — not inside the Jupyter kernel. Open **three terminals** and run the following, in order, waiting for each to stabilize before starting the next.

**Terminal 1 — Driver + Controllers**: brings up the EtherCAT driver, `joint_state_broadcaster`, `joint_trajectory_controller`, and `gripper_controller` for the Dobot CR3A.

**Terminal 2 — MoveIt 2 Planner**: starts the motion planning pipeline (collision checking, IK, trajectory generation).

**Terminal 3 — Trajectory API Bridge**: exposes the simple `/api/move/joint_absolute` service that Step 14 below calls.

## Step 14: Send an Absolute Joint Move (Proprietary Trajectory Control)

With all three terminals from Step 13 running, send a single absolute joint-space target through the Trajectory API Bridge. This exercises the full **Proprietary Trajectory Control** path: `MoveJoints` service → MoveIt 2 planner → `joint_trajectory_controller` → WMX3 C-Spline interpolation → EtherCAT.

In [ ]:
import rclpy
from rclpy.node import Node
from movensys_manipulator_moveit_config.srv import MoveJoints  # Service for absolute joint movements

# 1. Initialize ROS 2 communication context (guaranteed run once)
if not rclpy.ok():
    rclpy.init()

# 2. Create a temporary client node to send motion commands
node = Node('notebook4_joints_client')
client = node.create_client(MoveJoints, '/api/move/joint_absolute')

print("📡 Connecting to trajectory planning service...")
if client.wait_for_service(timeout_sec=5.0):
    # 3. Formulate the absolute target pose (Safe Stance Pose)
    req = MoveJoints.Request()
    req.joint_names = ['joint1', 'joint2', 'joint3', 'joint4', 'joint5', 'joint6']
    req.joint_values = [0.0, 0.704, -1.989, -0.287, 1.57, 0.0]  # Safe target angles in radians
    
    print("🎯 [EXECUTE] Sending absolute stance pose command to Dobot CR3A...")
    
    # 4. Call the service and wait for the motion block to complete
    future = client.call_async(req)
    rclpy.spin_until_future_complete(node, future)
    
    res = future.result()
    if res is not None:
        print(f"✅ [SUCCESS] Motion Execution: {res.success} | Message: {res.message}")
    else:
        print("❌ [ERROR] Service call failed during execution.")
else:
    print("❌ [TIMEOUT] Service '/api/move/joint_absolute' not found.")
    print("👉 Please verify that Terminal 2 (trajectory_service.launch.py) is running successfully.")

# Clean up the temporary node
node.destroy_node()


## Step 15: Compare with Standard ROS 2 Control (`wmx_r2_control`)

Now repeat the same "move to the safe stance pose" motion through the **second architecture**. Stop Terminal 1 from Step 13 and launch the `wmx_r2_control` variant instead, which loads WMX3 purely as a `ros2_control` hardware interface plugin behind the standard `controller_manager`:

Because this path is a *standard* `ros2_control` `joint_trajectory_controller`, there is no custom `MoveJoints` service — instead you talk to it through the standard `control_msgs/action/FollowJointTrajectory` action, the same interface Gazebo simulations and other ros2_control robots expose.

In [ ]:
import rclpy
from rclpy.action import ActionClient
from rclpy.node import Node
from control_msgs.action import FollowJointTrajectory
from trajectory_msgs.msg import JointTrajectoryPoint
from builtin_interfaces.msg import Duration

if not rclpy.ok():
    rclpy.init()

node = Node('notebook4_ros2_control_client')
action_client = ActionClient(node, FollowJointTrajectory, '/joint_trajectory_controller/follow_joint_trajectory')

print("📡 Connecting to standard ros2_control action server...")
if action_client.wait_for_server(timeout_sec=5.0):
    goal = FollowJointTrajectory.Goal()
    goal.trajectory.joint_names = ['joint1', 'joint2', 'joint3', 'joint4', 'joint5', 'joint6']

    point = JointTrajectoryPoint()
    point.positions = [0.0, 0.704, -1.989, -0.287, 1.57, 0.0]  # same safe stance pose as Step 14
    point.time_from_start = Duration(sec=3)
    goal.trajectory.points = [point]

    print("🎯 [EXECUTE] Sending FollowJointTrajectory goal (standard ros2_control path)...")
    send_future = action_client.send_goal_async(goal)
    rclpy.spin_until_future_complete(node, send_future)

    goal_handle = send_future.result()
    if goal_handle is not None and goal_handle.accepted:
        result_future = goal_handle.get_result_async()
        rclpy.spin_until_future_complete(node, result_future)
        print(f"✅ [SUCCESS] Trajectory execution finished: {result_future.result().result}")
    else:
        print("❌ [REJECTED] The controller rejected the trajectory goal.")
else:
    print("❌ [TIMEOUT] Action server '/joint_trajectory_controller/follow_joint_trajectory' not found.")
    print("👉 Please verify Terminal 1 is running wmx_r2_control_cr3a_manipulator.launch.py (Step 15, not Step 13).")

node.destroy_node()

---
✅ You have now driven the same manipulator motion through both architectures. Continue to `05_Visualization_and_Data_Logging.ipynb` to log and visualize joint feedback from either path (or the Gazebo simulation).